# ===============================
# 0. Install dependencies
# ===============================


In [ ]:
# !pip install openai  
# !pip install faiss-cpu
# !pip install pandas 
# !pip install numpy 
# !pip install tqdm
# !pip install python-dotenv

import os
import pandas as pd
import numpy as np
import sqlite3
import dotenv
from openai import OpenAI
from tqdm import tqdm
import re
import faiss
from pathlib import Path
from dotenv import load_dotenv

# Laad de .env-variabelen
load_dotenv()

# Haal de API key op
api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("✅ API key gevonden!")
else:
    print("❌ Geen API key gevonden. Controleer of .env bestaat en de juiste sleutel bevat.")

client = OpenAI(api_key=api_key) 



# ===============================
# 1. Load CSVs
# ===============================

In [ ]:
DATA_DIR = "text-input"
all_files = sorted(Path(DATA_DIR).glob("*.csv"))

dfs = []
for f in all_files:
    inv_nr = f.stem  # bv. "9221" of "9221a"
    df_temp = pd.read_csv(f, sep=",")
    df_temp.columns = ["filename", "text"]
    df_temp["inv_nr"] = inv_nr
    dfs.append(df_temp)

df_all = pd.concat(dfs, ignore_index=True)
print(f"Loaded {len(df_all)} rows from {len(all_files)} files")


# ===============================
# 2. Optional: spelling normalization
# ===============================

In [ ]:
def normalize_text(text):
    if not isinstance(text, str):
        return ""
    text = text.replace("„", '"').replace("’", "'")
    text = re.sub(r"[:;]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

APPLY_NORMALIZATION = True

if APPLY_NORMALIZATION:
    df_all["text"] = df_all["text"].apply(normalize_text)



# ===============================
# 3. Split text into overlapping chunks
# ===============================

In [ ]:
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

def chunk_over_paginagrenzen(df_inv, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """
    Maakt doorlopende chunks over paginagrenzen heen voor één inventarisnummer.
    
    df_inv: dataframe met kolommen ['filename', 'text'] voor 1 inv_nr, pagina's in juiste volgorde
    chunk_size: aantal woorden per chunk
    overlap: aantal woorden overlap tussen opeenvolgende chunks
    
    Geeft terug:
        chunks: lijst met strings
        chunk_meta: lijst met dicts {'start_page':..., 'pages':[...]}
    """
    all_words = []
    page_boundaries = []  # tuples (start_idx, filename)
    word_count = 0
    
    for idx, row in df_inv.iterrows():
        words = str(row["text"]).split()
        all_words.extend(words)
        page_boundaries.append((word_count, row["filename"]))
        word_count += len(words)
    
    chunks = []
    chunk_meta = []
    start = 0
    while start < len(all_words):
        end = min(start + chunk_size, len(all_words))
        chunk_words = all_words[start:end]
        chunks.append(" ".join(chunk_words))
        
        # bepaal startpagina en alle pagina's in de chunk
        pages_in_chunk = [p for idx_p, p in page_boundaries if start <= idx_p < end]
        if not pages_in_chunk:
            pages_in_chunk = [df_inv.iloc[-1]["filename"]]
        
        chunk_meta.append({
            "start_page": pages_in_chunk[0],
            "pages": pages_in_chunk
        })
        
        start += chunk_size - overlap
    
    return chunks, chunk_meta

# Process each inventory number separately and store properly
all_chunks_data = []

for inv_nr in df_all['inv_nr'].unique():
    df_inv = df_all[df_all['inv_nr'] == inv_nr].copy()
    chunks, chunk_meta = chunk_over_paginagrenzen(df_inv, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)
    
    # Store with correct inv_nr
    for i, meta in enumerate(chunk_meta):
        all_chunks_data.append({
            'inv_nr': inv_nr,
            'chunk_id': i,
            'text': chunks[i],
            'start_page': meta['start_page'],
            'pages': meta.get('pages', [])
        })

print(f"Created {len(all_chunks_data)} chunks across {df_all['inv_nr'].nunique()} inventory numbers")


In [ ]:
print(chunk_meta[:3])  # voorbeeld metadata


# ===============================
# 4. Store chunks + metadata in SQLite
# ===============================

In [ ]:
DB_FILE = "text-metadata-sqlite/voc_documents.db"
conn = sqlite3.connect(DB_FILE)
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS documents (
    inv_nr TEXT,
    start_page TEXT,
    pages TEXT,
    chunk_id INTEGER,
    text TEXT
)
""")
conn.commit()

# Insert all chunks with correct inv_nr
for chunk_data in all_chunks_data:
    pages_str = ";".join(chunk_data['pages']) if chunk_data['pages'] else ""
    cur.execute("""
        INSERT INTO documents (inv_nr, start_page, pages, chunk_id, text)
        VALUES (?, ?, ?, ?, ?)
        """, (
            str(chunk_data['inv_nr']), 
            str(chunk_data['start_page']), 
            pages_str, 
            int(chunk_data['chunk_id']), 
            str(chunk_data['text'])
        ))

conn.commit()
conn.close()
print(f"✅ Inserted {len(all_chunks_data)} chunks into database")

# ===============================
# 5. FAISS helpers per inv_nr
# ===============================

In [ ]:
EMB_DIR = Path("embeddings")
EMB_DIR.mkdir(exist_ok=True)

def embeddings_file(inv_nr):
    return EMB_DIR / f"{inv_nr}.index"

def load_or_create_embeddings(inv_nr):
    """
    Laadt bestaande FAISS-embeddings voor een inventarisnummer,
    of maakt ze aan als ze nog niet bestaan. 
    Haalt de chunks op uit de lokale SQLite-database.
    """
    db_path = Path("text-metadata-sqlite/voc_documents.db")
    emb_file = Path(f"embeddings/{inv_nr}.db")

    # Veilig verbinding openen
    with sqlite3.connect(db_path) as conn:
        df_chunks = pd.read_sql_query(
            f"SELECT * FROM documents WHERE inv_nr='{inv_nr}' ORDER BY chunk_id",
            conn
        )

    if df_chunks.empty:
        raise ValueError(f"⚠️ Geen data gevonden voor inv_nr {inv_nr} in de database.")

    # Verwijder lege of NaN-chunks
    df_chunks["text"] = df_chunks["text"].fillna("").astype(str)
    df_chunks = df_chunks[df_chunks["text"].str.strip() != ""]
    chunks_inv = df_chunks["text"].tolist()

    if not chunks_inv:
        raise ValueError(f"⚠️ Alle chunks voor {inv_nr} zijn leeg na opschonen.")

    # Controleer of embeddings al bestaan
    if emb_file.exists():
        faiss_idx = faiss.read_index(str(emb_file))
        print(f"✅ Loaded FAISS index for {inv_nr}")
    else:
        print(f"🧠 Computing embeddings for {inv_nr} ({len(chunks_inv)} chunks)...")

        embeddings = []
        batch_size = 50
        for i in tqdm(range(0, len(chunks_inv), batch_size)):
            batch = chunks_inv[i:i+batch_size]

            try:
                resp = client.embeddings.create(
                    model="text-embedding-3-large",
                    input=batch
                )
                batch_embeddings = [r.embedding for r in resp.data if hasattr(r, "embedding")]
                embeddings.extend(batch_embeddings)
            except Exception as e:
                print(f"⚠️ Fout bij batch {i}: {e}")
                continue

        if not embeddings:
            raise RuntimeError(f"❌ Geen embeddings aangemaakt voor {inv_nr}.")

        embeddings = np.array(embeddings, dtype="float32")

        dim = embeddings.shape[1]
        faiss_idx = faiss.IndexFlatL2(dim)
        faiss_idx.add(embeddings)

        emb_file.parent.mkdir(parents=True, exist_ok=True)
        faiss.write_index(faiss_idx, str(emb_file))
        print(f"💾 Saved FAISS index for {inv_nr} -> {emb_file}")

    return faiss_idx, chunks_inv

# ===============================
# 6. Query multiple inv_nrs
# ===============================

In [ ]:
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

def search_query(query_text, inv_nrs, verantwoording_top=5, top_k=None):
    """
    Voert een semantische zoekopdracht uit over 1 of meer inventarisnummers.
    Haalt bijbehorende paginametadata op uit SQLite.
    """
    all_results = []
    results_dir = Path("results")
    results_dir.mkdir(exist_ok=True)

    for inv_nr in inv_nrs:
        print(f"\n🔎 Searching in {inv_nr} ...")

        # Laad FAISS-index en chunks
        faiss_idx, chunks_inv = load_or_create_embeddings(inv_nr)
        if not chunks_inv:
            print(f"⚠️ Geen chunks gevonden voor {inv_nr}, overslaan.")
            continue

        # Query embedding
        q_emb = np.array(
            client.embeddings.create(model="text-embedding-3-large", input=query_text).data[0].embedding,
            dtype="float32"
        ).reshape(1, -1)

        # *** FIX: Ensure top_k doesn't exceed available chunks ***
        if top_k is None:
            top_k = len(chunks_inv)
        top_k = min(top_k, len(chunks_inv), faiss_idx.ntotal)  # Added faiss_idx.ntotal check

        # FAISS search
        D, I = faiss_idx.search(q_emb, top_k)
        scores = 1 / (1 + D.flatten())
        I = I.flatten()

        # --- Metadata ophalen uit SQLite
        with sqlite3.connect(DB_FILE) as conn:
            df_meta = pd.read_sql_query(
                f"SELECT inv_nr, start_page, pages, chunk_id, text FROM documents WHERE inv_nr='{inv_nr}' ORDER BY chunk_id",
                conn
            )

        # *** FIX: Filter out invalid indices ***
        valid_mask = I < len(df_meta)
        I = I[valid_mask]
        scores = scores[valid_mask]
        
        if len(I) == 0:
            print(f"⚠️ Geen geldige resultaten voor {inv_nr}")
            continue

        # --- Combineer chunks met resultaten
        df_res = df_meta.iloc[I].copy()
        df_res["similarity"] = scores

        # --- Verantwoording genereren voor topn
        actual_verantwoording_count = min(verantwoording_top, len(df_res))
        verantwoordingen = []

        for i in range(actual_verantwoording_count):
            chunk_text = df_res.iloc[i]["text"]
            try:
                prompt = (
                    f"Je bent een historicus gespecialiseerd in koloniale geschiedenis, specifiek de VOC, en van de koloniale, Aziatische context waarin VOC-documenten zijn geschreven, en in 17e en 18e-eeuws Nederlands.\n"
                    f"Beantwoord in één korte zin waarom deze tekst relevant is voor de vraag:\n\n"
                    f"Vraag: {query_text}\n\n"
                    f"Tekstfragment:\n{chunk_text[:2000]}\n\n"
                    f"Geef alleen de verantwoording, geen herhaling van de tekst."
                )
                resp = client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=120,
                    temperature=0.2,
                )
                reason = resp.choices[0].message.content.strip()
            except Exception as e:
                reason = f"(Fout bij genereren verantwoording: {e})"

            verantwoordingen.append(reason)

        # Initialiseer kolom met lege strings
        df_res["verantwoording"] = ""
        
        # Wijs verantwoordingen toe aan de eerste N rijen
        if verantwoordingen:
            df_res.iloc[:actual_verantwoording_count, df_res.columns.get_loc("verantwoording")] = verantwoordingen

        # --- Opslaan
        safe_query = "".join([c for c in query_text if c.isalnum() or c in "-_ "]).strip().replace(" ", "_")
        out_file = results_dir / f"{safe_query}-{inv_nr}.csv"
        df_res.to_csv(out_file, index=False)
        print(f"💾 Resultaten opgeslagen in: {out_file}")

        all_results.append(df_res)

    # Combineer alle resultaten
    if all_results:
        df_all = pd.concat(all_results, ignore_index=True)
        return df_all[["inv_nr", "start_page", "pages", "chunk_id", "similarity", "verantwoording", "text"]]
    else:
        print("⚠️ Geen resultaten gevonden.")
        return pd.DataFrame(columns=["inv_nr", "start_page", "pages", "chunk_id", "similarity", "verantwoording", "text"])

# ===============================
# 7. Usage
# ===============================

In [ ]:
query_text = "{hier zoekvraag invullen}"
df_results = search_query(query_text, inv_nrs=["1267","2448"], verantwoording_top=5)